In [95]:
import numpy as np 
import pandas as pd

In [96]:
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

In [97]:
def show_basic_stats(movies_df, ratings_df):
    print('Количество фильмов:', len(movies_df['movieId']))
    print('Количество пользователей:', ratings_df['userId'][100835])
    print('Количество оценок:', len(ratings_df['userId']))
    print('Средний рейтинг по всем фильмам:', (sum(ratings_df['rating']))/len(ratings_df['userId']))

    pass

def find_popular_movies(movies_df, ratings_df, top_n=10):
    rating_count = ratings_df.groupby('movieId')['rating'].count().reset_index()
    rating_count = rating_count.rename(columns={'rating': 'ratings_count'})
    rating_count = rating_count.sort_values('ratings_count', ascending=False)
    top_films = rating_count.head(top_n)
    popular_movies = top_films.merge(movies_df, on='movieId')
    return popular_movies


In [ ]:
def create_user_movie_matrix(ratings_df):
    matrix = ratings_df.pivot_table(
        index='userId',      
        columns='movieId',   
        values='rating',     
        fill_value=0       
    )
    return matrix
matrix = create_user_movie_matrix(ratings)
def get_similar_movies(movie_id, similarity_df, movies, n=10):
    similar_movies = similarity_df[movie_id].sort_values(ascending=False)[1:n+1]
    similar_movies_info = movies[movies['movieId'].isin(similar_movies.index)]
    return similar_movies_info

def calculate_movie_similarity(user_movie_matrix):
    from sklearn.metrics.pairwise import cosine_similarity
    
    movie_vectors = user_movie_matrix.T
    
    # Вычисляем косинусное сходство между фильмами
    similarity = cosine_similarity(movie_vectors)
    similarity_df = pd.DataFrame(
        similarity,
        index=movie_vectors.index,  # movieId как индексы
        columns=movie_vectors.index  # movieId как столбцы
    )
    return similarity_df

similarity_df = calculate_movie_similarity(matrix)

In [105]:
def demo_recommendation(user_id, ratings, movies, similarity_df):
    user_ratings = ratings[ratings['userId'] == user_id]
    
    if len(user_ratings) == 0:
        return "У пользователя нет оценок"
    
    favorite_movie = user_ratings.loc[user_ratings['rating'].idxmax()]
    
    recommendations = get_similar_movies(
        favorite_movie['movieId'], 
        similarity_df, 
        movies
    )
    
    return recommendations

In [108]:
user_with_ratings = ratings['userId'].iloc[0]  # первый пользователь в данных

# Получи рекомендации
recommendations = demo_recommendation(
    user_id=user_with_ratings,
    ratings=ratings,
    movies=movies, 
    similarity_df=similarity_df
)

print("Рекомендации для пользователя:", user_with_ratings)
print(recommendations[['movieId', 'title', 'genres']])

Рекомендации для пользователя: 1
      movieId                                              title  \
46         50                         Usual Suspects, The (1995)   
97        110                                  Braveheart (1995)   
254       293  Léon: The Professional (a.k.a. The Professiona...   
257       296                                Pulp Fiction (1994)   
277       318                   Shawshank Redemption, The (1994)   
314       356                                Forrest Gump (1994)   
418       480                               Jurassic Park (1993)   
510       593                   Silence of the Lambs, The (1991)   
828      1089                              Reservoir Dogs (1992)   
2226     2959                                  Fight Club (1999)   

                                genres  
46              Crime|Mystery|Thriller  
97                    Action|Drama|War  
254        Action|Crime|Drama|Thriller  
257        Comedy|Crime|Drama|Thriller  
277          

In [ ]:
def pretty_demo():
    print("=== ДЕМОНСТРАЦИЯ РЕКОМЕНДАТЕЛЬНОЙ СИСТЕМЫ ===")
    print()
    
    popular = find_popular_movies(movies, ratings, 5)
    print("Самые популярные фильмы:")
    for i, row in popular.iterrows():
        print(f"{i+1}. {row['title']} ({row['ratings_count']} оценок)")  # Изменил 'rating' на 'ratings_count'
    print()
    
    # Покажи рекомендации для нескольких пользователей
    test_users = ratings['userId'].unique()[:3]  # первые 3 пользователя
    
    for user_id in test_users:
        recs = demo_recommendation(user_id, ratings, movies, similarity_df)
        if isinstance(recs, str):
            print(f"Пользователь {user_id}: {recs}")
        else:
            print(f"Рекомендации для пользователя {user_id}:")
            for i, row in recs.iterrows():
                print(f"   - {row['title']} ({row['genres']})")
        print()

# Запусти демонстрацию
pretty_demo()

=== ДЕМОНСТРАЦИЯ РЕКОМЕНДАТЕЛЬНОЙ СИСТЕМЫ ===

Самые популярные фильмы:
1. Forrest Gump (1994) (329 оценок)
2. Shawshank Redemption, The (1994) (317 оценок)
3. Pulp Fiction (1994) (307 оценок)
4. Silence of the Lambs, The (1991) (279 оценок)
5. Matrix, The (1999) (278 оценок)

Рекомендации для пользователя 1:
   - Usual Suspects, The (1995) (Crime|Mystery|Thriller)
   - Braveheart (1995) (Action|Drama|War)
   - Léon: The Professional (a.k.a. The Professional) (Léon) (1994) (Action|Crime|Drama|Thriller)
   - Pulp Fiction (1994) (Comedy|Crime|Drama|Thriller)
   - Shawshank Redemption, The (1994) (Crime|Drama)
   - Forrest Gump (1994) (Comedy|Drama|Romance|War)
   - Jurassic Park (1993) (Action|Adventure|Sci-Fi|Thriller)
   - Silence of the Lambs, The (1991) (Crime|Horror|Thriller)
   - Reservoir Dogs (1992) (Crime|Mystery|Thriller)
   - Fight Club (1999) (Action|Crime|Drama|Thriller)

Рекомендации для пользователя 2:
   - Harold and Kumar Go to White Castle (2004) (Adventure|Comedy)
   -